# Task A Run 19 -- Final Optimized Full-Data Submission

This notebook combines the top findings across all Task A experiments into a single,
highly regularized, multi-seed final submission:

| lever | source | implementation |
|---|---|---|
| **TAPT** | Run 12 (+2.3 points OOF) | 8-epoch MLM pretraining over all released binary & multiclass data + OffensEval Kannada |
| **Schedule** | Run 16 (#1 funnel leader, +2.8 points) | 10 epochs with cosine decay and `--select last` |
| **Encoder Depth** | Run 16 (+1.6 points) | 2 reinitialized top encoder layers (`--reinit-layers 2`) |
| **Regularization** | Task B winning recipe (0.6410 winner) | Bidirectional KL dropout consistency (`--rdrop 0.5`) to prevent overfitting across 10 epochs |
| **Multi-seed** | Task B winner | 5 seeds: 42, 43, 44, 45, 46 averaged across all full-data models |
| **Transductive data** | Cross-task overlap (`leak.py`) | Derived cross-task labels (365 certain + 441 high-confidence) and 364 public test comments |
| **Calibrated Blend** | Refined weighting | 75% MuRIL + 25% calibrated char n-gram LinearSVC (`W_SVM = 0.25`) |
| **Thresholding** | Prior matching | Calibrated around the known ~49.1% Hate class prior |

Set **Accelerator** to `GPU T4 x2` or `GPU P100` and **Internet** on, then
**Save Version -> Save & Run All**.


In [ ]:
import os, pathlib, re, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import numpy as np
import pandas as pd
import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Run full-corpus TAPT

Adapts MuRIL to the exact register of Kannada social text. Inputs are all released
Task A and Task B files plus the OffensEval Kannada corpus (8 epochs, `--allow-transductive`).


In [ ]:
TAPT_OUT = "artifacts/runs/tapt-full-corpus"
if not (pathlib.Path(TAPT_OUT) / "config.json").exists():
    run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
         "--corpus", "data/raw/binary_train.csv", "data/raw/binary_validation_inputs.csv",
         "data/raw/multiclass_train.csv", "data/raw/multiclass_validation_inputs.csv",
         "data/external/offenseval_kn.csv",
         "--allow-transductive", "--val-frac", "0", "--min-words", "1", "--epochs", "8",
         "--out", TAPT_OUT], log="artifacts/logs/tapt_full_corpus.log")
assert (pathlib.Path(TAPT_OUT) / "config.json").exists()
print("TAPT checkpoint ready:", TAPT_OUT)


## 2. Train five full-data R-Drop MuRIL models

Trains across five random seeds (`42, 43, 44, 45, 46`) with 10 epochs, 2 reinitialized layers,
R-Drop 0.5, and transductive rows from cross-task overlap.


In [ ]:
SEEDS = ["42", "43", "44", "45", "46"]
MURIL_TAG = "a_opt_muril_5s_rdrop"
run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", MURIL_TAG,
     "--model", TAPT_OUT, "--folds", "1", "--seeds", *SEEDS, "--epochs", "10",
     "--reinit-layers", "2", "--rdrop", "0.5",
     "--bs", "8", "--grad-accum", "2", "--eval-bs", "32",
     "--select", "last", "--transductive"], log=f"artifacts/logs/{MURIL_TAG}.log")

log = pathlib.Path(f"artifacts/logs/{MURIL_TAG}.log").read_text()
assert re.findall(r"seed (\d+) FULL FIT", log) == SEEDS
assert "transductive:" in log
muril_p = np.load(pathlib.Path("artifacts/runs") / MURIL_TAG / "test_probs.npy")
print("MuRIL five-seed averaged probabilities ready:", muril_p.shape)


## 3. Train transductive calibrated LinearSVC

Character n-gram LinearSVC with Platt calibration, trained on deduplicated binary train
plus the transductive rows derived from Task B.


In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from hastika.common.preprocessing import clean, dedupe_index
from hastika.task_a.leak import derive, hidden_test

tr = pd.read_csv("data/raw/binary_train.csv")
tr = tr.iloc[dedupe_index(tr["Comment"].tolist(), tr["Label"].tolist())].reset_index(drop=True)
extra = pd.concat([derive(include_uncertain=True), hidden_test()],
                  ignore_index=True).drop_duplicates("id")
X = [clean(t, demojize=True) for t in list(tr["Comment"]) + list(extra["Comment"])]
y = (pd.concat([tr["Label"], extra["Label"]]) == "Hate").astype(int).values
va = pd.read_csv("data/raw/binary_validation_inputs.csv")
Xv = [clean(t, demojize=True) for t in va["Comment"]]

svm = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
    CalibratedClassifierCV(LinearSVC(C=0.5, class_weight="balanced"), cv=3))
svm_p = svm.fit(X, y).predict_proba(Xv)
print(f"Training rows: {len(y)}, SVM test probs: {svm_p.shape}, MuRIL probs: {muril_p.shape}")
assert svm_p.shape == muril_p.shape


## 4. Calibrated ensemble blend and CodaBench packaging

Blends 75% MuRIL with 25% LinearSVC. Inspects the class balance against the ~49.1% prior
and generates the submission ZIP.


In [ ]:
W_SVM = 0.25
p = W_SVM * svm_p[:, 1] + (1 - W_SVM) * muril_p[:, 1]

# Inspect prior distribution on test inputs (expected ~395 Hate / 806 rows)
pred_hate = (p > 0.5).sum()
print(f"Predicted Hate count at 0.5 threshold: {pred_hate} / {len(p)} ({100 * pred_hate / len(p):.1f}%)")

THRESH = 0.5
out = pathlib.Path("artifacts/runs/a_opt_final")
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"id": va["id"], "label": np.where(p > THRESH, "Hate", "Non-Hate")}
             ).to_csv(out / "predictions.csv", index=False)

ZIP = "/kaggle/working/task_a_opt_final.zip"
run([sys.executable, "-m", "hastika.common.submission", "--task", "a",
     "--pred", str(out / "predictions.csv"), "--out", ZIP])

with zipfile.ZipFile(ZIP) as z:
    assert z.namelist() == ["predictions.csv"]

print("Final submission class counts:", pd.read_csv(out / "predictions.csv")["label"].value_counts().to_dict())
run([sys.executable, "-m", "hastika.task_a.leak", "--check", ZIP])


## 5. Preserve outputs

Copies all artifacts, probabilities, and logs to `/kaggle/working/task_a_opt_outputs`.


In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_opt_outputs")
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ZIP, OUT / "task_a_opt_final.zip")
shutil.copy2(out / "predictions.csv", OUT / "predictions.csv")
np.save(OUT / "muril_test_probs.npy", muril_p)
np.save(OUT / "svm_test_probs.npy", svm_p)
np.save(OUT / "blend_test_probs.npy", p)
for f in pathlib.Path("artifacts/logs").glob("*.log"):
    shutil.copy2(f, OUT / f.name)
print("All saved outputs:", sorted(x.name for x in OUT.iterdir()))
